In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import joblib
# Load dataset
movies = pd.read_csv("csvs/tmdb_5000_movies.csv")

# Keep useful columns
movies = movies[["title", "overview", "genres"]]

# Fill NaN values
movies["overview"] = movies["overview"].fillna("")
movies["genres"] = movies["genres"].fillna("")

# Combine text features
movies["content"] = movies["overview"] + " " + movies["genres"]
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["content"])

print("Shape of TF-IDF matrix:", tfidf_matrix.shape)
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Map movie title → index
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()
def recommend_movies(title, num_recommendations=5):
    # Get index of movie
    idx = indices.get(title)
    if idx is None:
        return ["Movie not found!"]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top N movies (excluding itself)
    sim_scores = sim_scores[1:num_recommendations+1]
    movie_indices = [i[0] for i in sim_scores]

    return movies["title"].iloc[movie_indices].tolist()
print(recommend_movies("Inception", 5))
print(recommend_movies("The Dark Knight", 5))

# Save vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Save similarity matrix
joblib.dump(cosine_sim, "cosine_sim.pkl")

# Save index mapping
joblib.dump(indices, "indices.pkl")

# Save movies dataframe
movies.to_csv("movies_clean.csv", index=False)



Shape of TF-IDF matrix: (4803, 20988)
['The Helix... Loaded', 'Cypher', 'Mission: Impossible - Rogue Nation', 'Pandorum', 'The Fifth Element']
['The Dark Knight Rises', 'Batman Returns', 'Batman Forever', 'Batman: The Dark Knight Returns, Part 2', 'Batman Begins']
